### Scrape Jobs

Python Job Scraping library "JobSpy": https://github.com/speedyapply/JobSpy

In [41]:
from pathlib import Path
from jobspy import scrape_jobs
import pandas as pd
from IPython.display import HTML, display

jobs = scrape_jobs(
    site_name=["indeed", "linkedin"],  # zip_recruiter, glassdoor, google, bayt, bdjobs
    linkedin_fetch_description=True,
    search_term='"AI Engineer"',
    location="USA",
    country_indeed="USA",
    job_type="fulltime",
    hours_old=240,
    results_wanted=10,
)

df = pd.DataFrame(jobs)
print(f"Total jobs scraped: {len(df)}")
print(df[["title", "company", "site"]])

Total jobs scraped: 20
                                             title                    company  \
0                            Senior ML/AI Engineer                    SANDBAR   
1                                    Data Engineer                    MANTECH   
2   AI (Artificial Intelligence) Engineer (Remote)  NexGen Technologies, Inc.   
3                                      AI Engineer                        NaN   
4                Lead AI Engineer (AI Foundations)                Capital One   
5              Oracle AI Engineer - Senior Manager                        PwC   
6              Oracle AI Engineer - Senior Manager                        PwC   
7              Oracle AI Engineer - Senior Manager                        PwC   
8              Oracle AI Engineer - Senior Manager                        PwC   
9              Oracle AI Engineer - Senior Manager                        PwC   
10                                     AI Engineer                 InterImage   
11   

### Filter

In [43]:
# We ensure the title contains BOTH "AI" and "Engineer"
# We need to do this, because for indeed, also the job descriptions are searched,
# so we could get job that only have "AI Engineer" in the description, but not in the title.
mask = df["title"].str.contains("AI", case=False, na=False) & df["title"].str.contains(
    "Engineer", case=False, na=False
)
matching_jobs = df[mask].copy()

# We drop duplicates based on the combination of 'title' and 'company'
filtered_jobs = matching_jobs.drop_duplicates(subset=["title", "company"]).copy()

print(f"Total jobs found: {len(df)}")
print(f"Jobs matching 'AI' + 'Engineer' in title: {len(matching_jobs)}")
print(f"Jobs after deduplicating identical title/company pairs: {len(filtered_jobs)}")

Total jobs found: 20
Jobs matching 'AI' + 'Engineer' in title: 19
Jobs after deduplicating identical title/company pairs: 14


### Save the jobs as JSONL file

In [44]:
# Save the filtered jobs to a JSONL file in the "jobs" directory
# https://jsonlines.org/
output_dir = Path("jobs")
jsonl_path = output_dir / "1-scraped_jobs.jsonl"
filtered_jobs.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

### Display jobs

In [45]:
results_to_show = filtered_jobs[["title", "company", "job_url"]].copy()
results_to_show["job_url"] = results_to_show["job_url"].apply(
    lambda url: f'<a href="{url}" target="_blank">{url}</a>' if pd.notna(url) else ""
)

display(HTML(results_to_show.to_html(escape=False, index=False)))

title,company,job_url
Senior ML/AI Engineer,SANDBAR,https://www.indeed.com/viewjob?jk=c146335910a915d7
AI (Artificial Intelligence) Engineer (Remote),"NexGen Technologies, Inc.",https://www.indeed.com/viewjob?jk=b8b25b4aaf482123
AI Engineer,NaN,https://www.indeed.com/viewjob?jk=c3d5ae2f3753dbff
Lead AI Engineer (AI Foundations),Capital One,https://www.indeed.com/viewjob?jk=85bdd2fdebefde9a
Oracle AI Engineer - Senior Manager,PwC,https://www.indeed.com/viewjob?jk=479942b34465086e
AI Engineer,InterImage,https://www.linkedin.com/jobs/view/4395309456
AI Engineer,SoFi,https://www.linkedin.com/jobs/view/4393576713
Python AI Engineer,Arka Innovate,https://www.linkedin.com/jobs/view/4392455560
AI Engineer,Analytica,https://www.linkedin.com/jobs/view/4395041804
AI Engineer,Oteemo Inc.,https://www.linkedin.com/jobs/view/4393208617
